# AI Model Validation: From Mathematics to Trust
## Episode 09 — Probability Calibration: Can We Trust the Model's Confidence?

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

A model can classify correctly and still be dangerously wrong about how certain it is.

> **When a model says 90%, does 90% actually mean 90%?**

This episode develops probability semantics, calibration, discrimination versus calibration, reliability diagrams, Brier score, log loss, calibration error, overconfidence, underconfidence, recalibration, temperature scaling, calibration uncertainty, distribution shift and monitoring.

> **Confidence is a claim. Calibration tests that claim.**


## 1. Probability Is a Quantitative Claim

For binary outcome Y:

### Y ∈ {0, 1}

A probabilistic classifier returns:

### p̂(x) = P̂(Y = 1 | X = x)

Ideal calibration means:

### P(Y = 1 | p̂ = p) = p

If many comparable cases receive probability 0.80, approximately 80% should experience the event under the population and conditions being evaluated.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import platform, sys
print("Python :",sys.version.split()[0])
print("System :",platform.system(),platform.release())
print("Machine:",platform.machine())
print("NumPy  :",np.__version__)


## 2. Confidence Is Not Correctness

For one case:

### p̂ = 0.95

does not guarantee Y=1. A 95% event can fail.

Calibration is a statistical property across predictions.

### High Confidence ≠ Guaranteed Correctness

### Correct Prediction ≠ Calibrated Probability


## 3. Controlled Probability Experiment


In [ ]:
rng=np.random.default_rng(909)
n=12000
x=rng.normal(0,1,n)

def sigmoid(z): return 1/(1+np.exp(-z))
def logit(p):
    p=np.clip(p,1e-12,1-1e-12)
    return np.log(p/(1-p))

true_p=sigmoid(1.5*x-.3)
y=rng.binomial(1,true_p)

p_cal=true_p.copy()
p_over=sigmoid(1.8*logit(true_p))
p_under=sigmoid(.55*logit(true_p))

print("N:",n)
print("Observed prevalence:",y.mean())
print("Mean true probability:",true_p.mean())


## 4. Same Ranking, Different Confidence

The overconfident and underconfident transformations are monotonic.

They preserve ordering while changing probability magnitude.

### Same Ranking ≠ Same Calibration

This separates discrimination from probability reliability.


## 5. Reliability Diagram

For probability bin Bₘ:

### Confidence(Bₘ) = (1 / |Bₘ|) Σᵢ∈Bₘ p̂ᵢ

### Observed Frequency(Bₘ) = (1 / |Bₘ|) Σᵢ∈Bₘ yᵢ

Perfect calibration lies near:

### Observed Frequency(Bₘ) = Confidence(Bₘ)


In [ ]:
def reliability(y,p,bins=12):
    edges=np.linspace(0,1,bins+1)
    mean_p=[]; observed=[]; counts=[]
    for lo,hi in zip(edges[:-1],edges[1:]):
        mask=(p>=lo)&(p<hi if hi<1 else p<=hi)
        if mask.sum():
            mean_p.append(p[mask].mean())
            observed.append(y[mask].mean())
            counts.append(mask.sum())
    return np.array(mean_p),np.array(observed),np.array(counts)

plt.figure(figsize=(7,6))
plt.plot([0,1],[0,1],linestyle="--",label="Ideal")
for label,p in [("Calibrated",p_cal),("Overconfident",p_over),("Underconfident",p_under)]:
    mp,obs,_=reliability(y,p)
    plt.plot(mp,obs,marker="o",label=label)
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed event frequency")
plt.title("Reliability Diagram")
plt.legend(); plt.grid(alpha=.25); plt.show()


## 6. Overconfidence and Underconfidence

Overconfidence means probability predictions are more extreme than justified by empirical outcomes.

Underconfidence means probability predictions are too conservative.

Both can matter operationally when probabilities control thresholds, escalation, abstention or risk limits.


## 7. Discrimination ≠ Calibration

Discrimination asks whether positive cases tend to receive higher scores than negative cases.

Calibration asks whether the probability values correspond to empirical frequencies.

A model may have excellent ranking and poor calibration.


In [ ]:
def auc_pairwise(y,p):
    pos=p[y==1]; neg=p[y==0]
    total=count=0
    for block in np.array_split(pos,20):
        total+=(block[:,None]>neg[None,:]).sum()
        total+=.5*(block[:,None]==neg[None,:]).sum()
        count+=len(block)*len(neg)
    return total/count

for label,p in [("Calibrated",p_cal),("Overconfident",p_over),("Underconfident",p_under)]:
    print(f"{label:<15} AUC={auc_pairwise(y,p):.6f}")


The AUC values remain essentially identical because ranking is preserved. Calibration, however, is visibly different.


## 8. Brier Score

For binary outcomes:

### Brier = (1/n) Σᵢ₌₁ⁿ (p̂ᵢ − yᵢ)²

Lower is better.

For an event with true probability q and reported probability p:

### E[(p − Y)²] = q(p − 1)² + (1 − q)p²

Expanding:

### E[(p − Y)²] = (p − q)² + q(1 − q)

The second term does not depend on reported p.

Therefore expected Brier loss is minimized at:

### p = q


In [ ]:
def brier(y,p): return np.mean((p-y)**2)
for label,p in [("Calibrated",p_cal),("Overconfident",p_over),("Underconfident",p_under)]:
    print(f"{label:<15} Brier={brier(y,p):.6f}")


## 9. Brier Decomposition

For grouped forecasts, Brier score can be decomposed conceptually:

### Brier = Reliability − Resolution + Uncertainty

Reliability measures calibration error.

Resolution rewards separating cases into groups with different observed event frequencies.

Uncertainty reflects intrinsic outcome variability associated with prevalence.


In [ ]:
def brier_decomposition(y,p,bins=15):
    edges=np.linspace(0,1,bins+1); base=y.mean(); n=len(y)
    rel=res=0.0
    for lo,hi in zip(edges[:-1],edges[1:]):
        mask=(p>=lo)&(p<hi if hi<1 else p<=hi)
        if mask.sum():
            w=mask.sum()/n
            pred=p[mask].mean(); obs=y[mask].mean()
            rel+=w*(pred-obs)**2
            res+=w*(obs-base)**2
    unc=base*(1-base)
    return rel,res,unc

for label,p in [("Calibrated",p_cal),("Overconfident",p_over),("Underconfident",p_under)]:
    rel,res,unc=brier_decomposition(y,p)
    print(f"{label:<15} reliability={rel:.5f} resolution={res:.5f} uncertainty={unc:.5f}")


Because continuous probabilities are grouped into bins, this empirical decomposition is approximate and depends on the grouping procedure.


## 10. Logarithmic Loss

### LogLoss = −(1/n) Σᵢ [yᵢ ln(p̂ᵢ) + (1−yᵢ) ln(1−p̂ᵢ)]

Log loss strongly penalizes confident incorrect probability predictions.


In [ ]:
def logloss(y,p):
    p=np.clip(p,1e-15,1-1e-15)
    return -np.mean(y*np.log(p)+(1-y)*np.log(1-p))

for label,p in [("Calibrated",p_cal),("Overconfident",p_over),("Underconfident",p_under)]:
    print(f"{label:<15} LogLoss={logloss(y,p):.6f}")

print("\nLoss for actual y=1:")
for p in [.9,.5,.1,.01,.001]:
    print(f"p={p:.3f} -> {-np.log(p):.4f}")


## 11. Proper Scoring Rules

Brier score and log loss are proper scoring rules.

In expectation, truthful probability reporting minimizes expected loss when the underlying probability is known.

### Classification Accuracy ≠ Probability Quality

Accuracy evaluates a thresholded decision. Proper scoring rules evaluate the probability forecast itself.


## 12. Expected Calibration Error — ECE

A commonly used empirical summary is:

### ECE = Σₘ (|Bₘ| / n) × |Observed(Bₘ) − Confidence(Bₘ)|

Lower is better, but ECE depends on binning and sample size.


In [ ]:
def ece(y,p,bins=15):
    mp,obs,count=reliability(y,p,bins)
    return np.sum((count/len(y))*np.abs(obs-mp))

for bins in [5,10,15,30,60]:
    print(f"bins={bins:>2} calibrated={ece(y,p_cal,bins):.5f} "
          f"overconfident={ece(y,p_over,bins):.5f}")


### ECE ≠ Binning-Invariant Truth

Changing the bins can change the reported ECE. The binning definition belongs in the evidence record.


## 13. Maximum Calibration Error

### MCE = maxₘ |Observed(Bₘ) − Confidence(Bₘ)|

MCE highlights the largest observed bin gap, but can be unstable when some bins contain few observations.


In [ ]:
def mce(y,p,bins=15):
    mp,obs,_=reliability(y,p,bins)
    return np.max(np.abs(obs-mp))

for label,p in [("Calibrated",p_cal),("Overconfident",p_over),("Underconfident",p_under)]:
    print(f"{label:<15} MCE={mce(y,p):.5f}")


## 14. Calibration-in-the-Large

A global diagnostic is:

### CITL Gap = mean(p̂) − mean(Y)

Ideal value is approximately zero.

This detects systematic global overprediction or underprediction but cannot expose every local calibration failure.


In [ ]:
for label,p in [("Calibrated",p_cal),("Overconfident",p_over),("Underconfident",p_under)]:
    gap=p.mean()-y.mean()
    print(f"{label:<15} mean(p)={p.mean():.5f} prevalence={y.mean():.5f} gap={gap:.5f}")


## 15. Calibration Intercept and Slope

A useful calibration relationship is:

### logit[P(Y=1)] = α + β logit(p̂)

Ideal calibration suggests approximately:

### α = 0
### β = 1

The intercept reflects systematic offset.

The slope reflects whether predictions are too extreme or too compressed, subject to model and sampling assumptions.


## 16. Calibration Is Population-Dependent

### CalibrationPopulation A ⇏ CalibrationPopulation B

Changes in prevalence or conditional relationships can invalidate an earlier probability scale.


## 17. Experiment — Calibration Under Prevalence Shift


In [ ]:
x_shift=rng.normal(0,1,8000)
deploy_true=sigmoid(1.5*x_shift+.55)
deploy_y=rng.binomial(1,deploy_true)

# Model still represents the earlier environment.
deploy_model=sigmoid(1.5*x_shift-.3)

print("Observed deployment prevalence:",deploy_y.mean())
print("Mean predicted probability     :",deploy_model.mean())
print("Brier:",brier(deploy_y,deploy_model))
print("ECE:",ece(deploy_y,deploy_model))

mp,obs,_=reliability(deploy_y,deploy_model)
plt.figure(figsize=(7,6))
plt.plot([0,1],[0,1],linestyle="--",label="Ideal")
plt.plot(mp,obs,marker="o",label="Shifted deployment")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed event frequency")
plt.title("Calibration Can Break Under Shift")
plt.legend(); plt.grid(alpha=.25); plt.show()


A probability scale that was meaningful during validation can become systematically wrong after the deployment environment changes.


## 18. Recalibration

A recalibration method learns:

### g : score → calibrated probability

so that:

### p̃ = g(s)

Recalibration can improve probability reliability without necessarily changing the underlying ranking.


## 19. Platt-Style Logistic Calibration

A parametric calibration mapping can be written:

### p̃ = sigmoid(a × logit(p̂) + b)

Parameters a and b are estimated using separate calibration data.


In [ ]:
def fit_calibrator(p,y,iterations=30):
    z=logit(p)
    X=np.column_stack([z,np.ones(len(z))])
    theta=np.array([1.0,0.0])
    for _ in range(iterations):
        q=sigmoid(X@theta)
        grad=X.T@(q-y)
        w=q*(1-q)
        H=X.T@(X*w[:,None])
        step=np.linalg.solve(H+1e-8*np.eye(2),grad)
        theta-=step
        if np.linalg.norm(step)<1e-9: break
    return theta

idx=rng.permutation(n)
cal_idx=idx[:4000]
test_idx=idx[4000:]

a,b=fit_calibrator(p_over[cal_idx],y[cal_idx])
p_recal=sigmoid(a*logit(p_over[test_idx])+b)

print("a,b:",a,b)
print("Before Brier :",brier(y[test_idx],p_over[test_idx]))
print("After Brier  :",brier(y[test_idx],p_recal))
print("Before LogLoss:",logloss(y[test_idx],p_over[test_idx]))
print("After LogLoss :",logloss(y[test_idx],p_recal))


## 20. Reliability Before and After Recalibration


In [ ]:
mp1,obs1,_=reliability(y[test_idx],p_over[test_idx])
mp2,obs2,_=reliability(y[test_idx],p_recal)

plt.figure(figsize=(7,6))
plt.plot([0,1],[0,1],linestyle="--",label="Ideal")
plt.plot(mp1,obs1,marker="o",label="Before")
plt.plot(mp2,obs2,marker="o",label="After")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed event frequency")
plt.title("Out-of-Sample Recalibration")
plt.legend(); plt.grid(alpha=.25); plt.show()


Calibration must be learned on data separate from final evaluation. Otherwise apparent improvement can be optimistic.


## 21. Temperature Scaling

For logit z:

### p̃ = sigmoid(z / T)

where T > 0.

If T > 1, probabilities become less extreme.

If 0 < T < 1, they become more extreme.

Positive scalar temperature scaling preserves ranking while changing confidence.


In [ ]:
z=logit(p_over[test_idx])
for T in [.5,1,1.5,2,3]:
    pT=sigmoid(z/T)
    print(f"T={T:<3} Brier={brier(y[test_idx],pT):.5f} LogLoss={logloss(y[test_idx],pT):.5f}")


In multiclass neural networks, temperature scaling is commonly applied to logits before softmax. A single temperature is simple but cannot repair every form of miscalibration.


## 22. Isotonic Calibration

Isotonic calibration learns a flexible non-decreasing mapping:

### s₁ ≤ s₂ ⇒ g(s₁) ≤ g(s₂)

It can represent nonlinear calibration relationships but may overfit when calibration data are limited.

### Flexibility ↔ Variance


## 23. Train, Calibrate, Test

A defensible workflow is:

### Training Data → Fit Model

### Calibration Data → Fit Calibration Mapping

### Test Data → Evaluate Final System

Repeatedly tuning against the test set destroys its role as untouched final evidence.


## 24. Calibration Has Sampling Uncertainty

For a bin with nₘ independent Bernoulli outcomes and observed rate q̂ₘ:

### SE(q̂ₘ) ≈ √[q̂ₘ(1 − q̂ₘ) / nₘ]

A calibration point based on 20 cases is weaker evidence than one based on 20,000 cases.


In [ ]:
mp,obs,count=reliability(y,p_over,12)
for pred,rate,n_bin in zip(mp,obs,count):
    se=np.sqrt(rate*(1-rate)/n_bin)
    print(f"mean p={pred:.3f}, observed={rate:.3f}, n={n_bin:4d}, approx SE={se:.4f}")


## 25. Rare Events and Sparse Probability Regions

In rare-event systems, the high-probability region may contain very few observations.

Large total sample size does not guarantee strong calibration evidence in the region where important decisions occur.

### Global Sample Size ≠ Local Evidence Strength


## 26. Global and Local Calibration

A model can appear globally calibrated while failing within a subgroup, time period, probability region or deployment context.

### Global Calibration ≠ Local Calibration

Validation may need calibration evidence by:

- probability region
- subgroup
- geography
- time
- model version
- use case
- operating threshold


## 27. Calibration Drift

At deployment time t:

### Calibrationₜ(p) = Pₜ(Y=1 | p̂=p)

Environmental change can produce:

### Calibrationₜ₀(p) ≠ Calibrationₜ₁(p)

Calibration is therefore monitorable evidence, not a permanent certificate.


## 28. Calibration and Selective Prediction

Suppose automation acts only when probability exceeds a confidence requirement and otherwise abstains.

If confidence is miscalibrated, the abstention rule itself can be unreliable.

### Calibration → Threshold Meaning → Decision Reliability


## 29. What Calibration Does Not Establish

A calibrated model is not automatically:

- sufficiently accurate
- discriminative
- robust
- safe
- stable under shift
- causally correct
- appropriate for intended use

### Calibration ≠ Complete Validation

It is one dimension of model evidence.


## 30. OpenVals Perspective

A serious probabilistic validation platform should preserve:

- raw probability outputs
- calibration and test datasets
- prevalence
- reliability curves
- bin definitions and counts
- Brier score
- log loss
- ECE and MCE
- calibration-in-the-large
- calibration slope/intercept where appropriate
- recalibration method
- before/after calibration evidence
- subgroup calibration
- temporal calibration drift
- uncertainty around calibration estimates

The validation question is not simply:

> How confident is the model?

It is:

> **Does the model's confidence deserve to be interpreted as probability?**


## 31. Scientific Validation Chain

### Score → Probability Claim → Calibration Test → Proper Scoring Rule → Uncertainty → Recalibration → Monitoring → Evidence


## 32. Mathematical Summary

### P(Y=1 | p̂=p) = p

### Brier = (1/n) Σ(p̂ᵢ − yᵢ)²

### E[(p−Y)²] = (p−q)² + q(1−q)

### Brier = Reliability − Resolution + Uncertainty

### LogLoss = −(1/n) Σ[yᵢln(p̂ᵢ) + (1−yᵢ)ln(1−p̂ᵢ)]

### ECE = Σₘ (|Bₘ|/n)|Observed(Bₘ)−Confidence(Bₘ)|

### MCE = maxₘ|Observed(Bₘ)−Confidence(Bₘ)|

### logit[P(Y=1)] = α + βlogit(p̂)

### p̃ = sigmoid(a logit(p̂) + b)

### p̃ = sigmoid(z/T)

### SE(q̂ₘ) ≈ √[q̂ₘ(1−q̂ₘ)/nₘ]


## 33. Central Conclusion

### Model Confidence ≠ Probability Reliability

> **Confidence is a claim. Calibration tests that claim.**

A model can rank cases correctly and still assign misleading probabilities.

A model can be accurate and still be overconfident.

A probability scale can be calibrated today and fail after distribution shift tomorrow.

Calibration transforms “the model says 90%” into a scientifically testable statement.

The goal is not to make confidence look impressive. The goal is to determine whether confidence corresponds to empirical reality.

### Next Episode

**Episode 10 — Robustness and Stress Testing: What Happens When the World Changes?**

Episode 10 can move into perturbations, noise, missingness, covariate shift, subgroup stress tests, adversarial sensitivity, worst-case performance, robustness curves, degradation rates, and the distinction between average performance and resilient performance.
